# Phase 6: Three-Tier Memory System
## Working, Episodic, and Semantic Memory Integration

**Goal:** Integrate working memory, episodic memory, and semantic memory into a unified system with **zero catastrophic forgetting** at any tier.

### What this notebook does

1. Clone / pull FLUX repo from GitHub
2. Install dependencies + run `setup.py`
3. Initialize `PhaseLogger(phase=6)`, detect hardware, load `HF_TOKEN`
4. Load Phases 1–5 checkpoints + smoke test all components
5. Train Three-Tier Memory System (5 stages)
6. Save checkpoint + upload to HuggingFace Hub
7. Test 1: One-shot episodic write/read accuracy
8. Test 2: Semantic memory protection after 1000 writes
9. Test 3: Forgetting score = 0.0 across 10 task pairs
10. Demo 1: Cross-session memory (facts survive session boundary)
11. Demo 2: Consolidation process live (episodic → semantic)
12. Demo 3: Zero forgetting over 1000 tasks
13. View Results summary + full log
14. Final upload (logs → HF Hub + GitHub)
15. Save artifacts to Kaggle output

### Key Physics

- **Working Memory:** Rolling field window — your desk (session-scoped)
- **Episodic Memory:** FAISS vector store — your diary (permanent, one-shot write)
- **Semantic Memory:** Protected field core — your deep skills (consolidation-only updates)
- **Consolidation:** Episodic → Semantic distillation (like sleep consolidation)
- **Zero forgetting:** New attractors form without destroying old ones

### Secrets Required

- `HF_TOKEN`: HuggingFace API token (Kaggle → Add-ons → Secrets)

---
## Cell 1: Clone / Pull Repository

In [ ]:
import os
import sys
import subprocess
import importlib
from pathlib import Path

REPO_URL = "https://github.com/Unseengap/FLUX.git"
WORK_DIR = Path("/kaggle/working/FLUX")

# ─────────────────────────────────────────────
# 1. Clone or Pull — always get the absolute
#    latest code from GitHub.
# ─────────────────────────────────────────────
if WORK_DIR.exists() and (WORK_DIR / '.git').exists():
    print("ℹ Repo already exists — pulling latest changes...")

    subprocess.run(
        ['git', 'remote', 'set-url', 'origin', REPO_URL],
        cwd=str(WORK_DIR), capture_output=True, text=True,
    )

    subprocess.run(['git', 'checkout', '.'], cwd=str(WORK_DIR),
                   capture_output=True, text=True)
    subprocess.run(['git', 'clean', '-fd'], cwd=str(WORK_DIR),
                   capture_output=True, text=True)

    fetch = subprocess.run(['git', 'fetch', '--all'], cwd=str(WORK_DIR),
                           capture_output=True, text=True)
    if fetch.returncode != 0:
        print(f"  ⚠ git fetch failed: {fetch.stderr.strip()}")
    result = subprocess.run(
        ['git', 'reset', '--hard', 'origin/main'],
        cwd=str(WORK_DIR), capture_output=True, text=True,
    )
    print(result.stdout.strip() or result.stderr.strip())

    head = subprocess.run(
        ['git', 'log', '--oneline', '-3'],
        cwd=str(WORK_DIR), capture_output=True, text=True,
    )
    print(f"\n  Latest commits:\n{head.stdout.strip()}")
    print("\n✓ Pulled & reset to latest origin/main")
else:
    if WORK_DIR.exists():
        import shutil
        shutil.rmtree(str(WORK_DIR))
    print(f"ℹ Cloning {REPO_URL}...")
    subprocess.run(['git', 'clone', REPO_URL, str(WORK_DIR)], check=True)
    print("✓ Cloned successfully")

os.chdir(str(WORK_DIR))
print(f"\nWorking directory: {os.getcwd()}")

# ─────────────────────────────────────────────
# 2. Flush cached Python modules so the kernel
#    picks up the freshly-pulled code.
# ─────────────────────────────────────────────
_stale = [m for m in sys.modules if any(
    m.startswith(p) for p in (
        'working_memory', 'episodic_memory', 'semantic_memory',
        'memory_router', 'consolidation', 'train_memory',
        'cgn', 'manifold', 'causal_graph', 'multi_timescale',
        'thermodynamic', 'temperature', 'energy_functions', 'online_learner',
        'cse', 'field', 'flux_utils', 'wave_types', 'interference',
        'attractor', 'field_ops', 'train_', 'demo_', 'test_',
        'gravity', 'mass_tracker', 'spatial_index', 'negative_mass',
    )
)]
for m in _stale:
    del sys.modules[m]
if _stale:
    print(f"  ✓ Flushed {len(_stale)} cached modules: {_stale[:5]}{'...' if len(_stale) > 5 else ''}")

# ─────────────────────────────────────────────
# 3. Delete stale Phase 6 checkpoint so training
#    runs fresh with the updated code.
# ─────────────────────────────────────────────
_stale_ckpt = WORK_DIR / 'checkpoints' / 'phase6.phase.pt'
if _stale_ckpt.exists():
    _stale_ckpt.unlink()
    print(f"  ✓ Deleted stale checkpoint: {_stale_ckpt.name}")

subprocess.run(['python', 'setup.py'], cwd=str(WORK_DIR),
               capture_output=True, text=True)

# Quick sanity check: verify Phase 6 modules are present
_wm_path = WORK_DIR / 'phases' / 'phase6' / 'working_memory.py'
_wm_text = _wm_path.read_text()
assert 'WorkingMemory' in _wm_text, "ERROR: working_memory.py missing WorkingMemory!"
print(f"  ✓ working_memory.py verified: WorkingMemory present")

_em_path = WORK_DIR / 'phases' / 'phase6' / 'episodic_memory.py'
_em_text = _em_path.read_text()
assert 'EpisodicMemory' in _em_text, "ERROR: episodic_memory.py missing EpisodicMemory!"
print(f"  ✓ episodic_memory.py verified: EpisodicMemory present")

_sm_path = WORK_DIR / 'phases' / 'phase6' / 'semantic_memory.py'
_sm_text = _sm_path.read_text()
assert 'SemanticMemory' in _sm_text, "ERROR: semantic_memory.py missing SemanticMemory!"
print(f"  ✓ semantic_memory.py verified: SemanticMemory present")

_mr_path = WORK_DIR / 'phases' / 'phase6' / 'memory_router.py'
_mr_text = _mr_path.read_text()
assert 'MemoryRouter' in _mr_text, "ERROR: memory_router.py missing MemoryRouter!"
print(f"  ✓ memory_router.py verified: MemoryRouter present")

_co_path = WORK_DIR / 'phases' / 'phase6' / 'consolidation.py'
_co_text = _co_path.read_text()
assert 'ConsolidationProcess' in _co_text, "ERROR: consolidation.py missing ConsolidationProcess!"
print(f"  ✓ consolidation.py verified: ConsolidationProcess present")

print("✓ setup.py refreshed")

---
## Cell 2: Install Dependencies & Setup

In [ ]:
!pip install -q datasets rich faiss-cpu huggingface_hub matplotlib
!python setup.py

---
## Cell 3: Initialize Logger, Detect Hardware & Load Secrets

In [ ]:
import sys, torch, platform, importlib
from pathlib import Path

# ── Add project paths ──
sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path.cwd() / 'phases' / 'phase1'))
sys.path.insert(0, str(Path.cwd() / 'phases' / 'phase2'))
sys.path.insert(0, str(Path.cwd() / 'phases' / 'phase3'))
sys.path.insert(0, str(Path.cwd() / 'phases' / 'phase4'))
sys.path.insert(0, str(Path.cwd() / 'phases' / 'phase5'))
sys.path.insert(0, str(Path.cwd() / 'phases' / 'phase6'))

# ── Force-reload all project modules ──
for _mod_name in list(sys.modules.keys()):
    if any(_mod_name.startswith(p) for p in (
        'flux_utils', 'working_memory', 'episodic_memory', 'semantic_memory',
        'memory_router', 'consolidation', 'train_memory',
        'cgn', 'manifold', 'causal_graph', 'multi_timescale',
        'thermodynamic', 'temperature', 'energy_functions', 'online_learner',
        'cse', 'field', 'wave_types', 'interference', 'attractor', 'field_ops',
        'gravity', 'mass_tracker', 'spatial_index', 'negative_mass',
    )):
        del sys.modules[_mod_name]

from flux_utils import (
    get_device, get_hardware_info, PhaseLogger, _resolve_hf_token,
    load_checkpoint, save_checkpoint, verify_checkpoint_chain,
    upload_checkpoint_to_hf, upload_logs_to_hf, git_commit_and_push,
    PhaseResults, checkpoint_exists,
)

# ── Initialize Phase 6 Logger ──
log = PhaseLogger(phase=6)
log.separator("Phase 6: Three-Tier Memory System")
log.cell_start("Cell 3 — Hardware & Secrets")

# ── Detect hardware ──
DEVICE = get_device()
hw = get_hardware_info()
log.info(f"Device: {DEVICE}")
log.info(f"PyTorch: {torch.__version__}")
log.info(f"Platform: {hw['platform']}")
for k, v in hw.items():
    print(f"  {k}: {v}")

if torch.cuda.is_available():
    log.info(f"GPU: {hw.get('gpu', 'N/A')}")
    log.info(f"VRAM: {hw.get('gpu_memory', 'N/A')}")
    log.info(f"CUDA: {hw.get('cuda', 'N/A')}")

# ── Load HuggingFace token ──
HF_TOKEN = _resolve_hf_token()
if HF_TOKEN:
    log.success("HuggingFace token loaded")
    print("  ✓ HF token loaded")
else:
    log.warning("No HuggingFace token — checkpoint upload will be skipped")
    print("  ⚠ No HF token — add HF_TOKEN to Kaggle Secrets for auto-upload")

log.cell_end("Cell 3 — Hardware & Secrets", "PASS")

---
## Cell 4: Load Phases 1–5 Checkpoints + Smoke Test

Each prior phase provides a component used by the memory system:
- **Phase 1 (CSE):** Encodes text → semantic waves (input to all memory tiers)
- **Phase 2 (ResonanceField):** The field that becomes semantic memory
- **Phase 4 (TL):** Thermodynamic settling for consolidation
- **Phase 5 (CGN):** Causal geometry nodes for reasoning over memories

In [ ]:
log.cell_start("Cell 4 — Load Phases 1–5 + Smoke Test")

import torch
import torch.nn.functional as F

# Force-reimport phase modules
import importlib
for _m in ['cse', 'field', 'thermodynamic', 'temperature', 'energy_functions',
           'online_learner', 'cgn', 'manifold', 'causal_graph', 'multi_timescale',
           'working_memory', 'episodic_memory', 'semantic_memory',
           'memory_router', 'consolidation']:
    if _m in sys.modules:
        importlib.reload(sys.modules[_m])

from cse import ContinuousSemanticEncoder
from field import ResonanceField, FIELD_H, FIELD_W, FIELD_D, FIELD_FEATURES
from working_memory import WorkingMemory
from episodic_memory import EpisodicMemory
from semantic_memory import SemanticMemory
from memory_router import MemoryRouter
from consolidation import ConsolidationProcess

# ── Load Phase 1 (CSE) ──
ckpt1 = load_checkpoint(1)
cse = ContinuousSemanticEncoder(**ckpt1.get('config', {}))
cse.load_state_dict(ckpt1['state_dict'])
cse = cse.to(DEVICE).eval()
for p in cse.parameters():
    p.requires_grad = False

# Smoke test CSE
test_wave = cse.encode("smoke test Phase 6 memory system")
assert test_wave.full.shape[-1] == 432, f"Bad wave dim: {test_wave.full.shape}"
assert not torch.isnan(test_wave.full).any(), "NaN in wave!"
log.success(f"Phase 1 CSE loaded: {sum(p.numel() for p in cse.parameters()):,} params")
print(f"  ✓ Phase 1 (CSE): wave shape {test_wave.full.shape}")

# ── Load Phase 2 (ResonanceField) ──
ckpt2 = load_checkpoint(2)
field_cfg = ckpt2.get('config', {}).get('field', {})
field = ResonanceField(**field_cfg)
field.load_state_dict(ckpt2['state_dict'])
field = field.to(DEVICE)

# Smoke test Field
vec = test_wave.full.mean(dim=0).to(DEVICE)
field_out, sims, locs = field.query(vec, k=4)
assert not torch.isnan(field_out).any(), "NaN in field query!"
log.success(f"Phase 2 Field loaded: {sum(p.numel() for p in field.parameters()):,} params")
print(f"  ✓ Phase 2 (ResonanceField): field {FIELD_H}×{FIELD_W}×{FIELD_D}×{FIELD_FEATURES}")

# ── Load Phase 5 (CGN) ──
ckpt5 = load_checkpoint(5)
log.success("Phase 5 CGN checkpoint loaded")
print(f"  ✓ Phase 5 (CGN): checkpoint loaded")

# ── Smoke test Phase 6 components ──
wave_dim = 432
feature_dim = 256

wm_test = WorkingMemory(window_size=32, wave_dim=wave_dim, feature_dim=feature_dim).to(DEVICE)
wm_test.add_perturbation(vec)
assert wm_test.size == 1, f"Working memory size should be 1, got {wm_test.size}"
log.success(f"WorkingMemory smoke test: {wm_test.size} entry")
print(f"  ✓ Phase 6 (WorkingMemory): smoke test OK")

em_test = EpisodicMemory(feature_dim=feature_dim)
with torch.no_grad():
    test_vec = wm_test.compress(vec.unsqueeze(0)).squeeze(0)
em_test.write(test_vec, fact="smoke test fact")
results = em_test.search(test_vec, k=1)
assert len(results) > 0, "Episodic search returned nothing!"
assert results[0][0].fact == "smoke test fact", f"Wrong fact: {results[0][0].fact}"
log.success("EpisodicMemory smoke test: write + search OK")
print(f"  ✓ Phase 6 (EpisodicMemory): write/search smoke test OK")

sm_test = SemanticMemory(field=field).to(DEVICE)
sm_test.protect_attractor(0)
assert sm_test.is_protected(0)
log.success("SemanticMemory smoke test: protection OK")
print(f"  ✓ Phase 6 (SemanticMemory): protection smoke test OK")

del wm_test, em_test, sm_test
torch.cuda.empty_cache() if torch.cuda.is_available() else None

print("\n  Phase 1 model: https://huggingface.co/UnseenGAP/FLUX (phase1.phase.pt)")
print("  Phase 2 model: https://huggingface.co/UnseenGAP/FLUX (phase2.phase.pt)")
print("  Phase 5 model: https://huggingface.co/UnseenGAP/FLUX (phase5.phase.pt)")
log.cell_end("Cell 4 — Load Phases 1–5 + Smoke Test", "PASS")

---
## Cell 5: Training / Load from Checkpoint

Five training stages:

- **Stage A:** Working memory — train compression + importance scoring
- **Stage B:** Episodic memory — one-shot write/read with FAISS
- **Stage C:** Semantic memory — field protection + consolidation
- **Stage D:** Memory router — cross-tier query integration
- **Stage E:** Zero forgetting — sequential task pair evaluation

**Estimated time:** ~5 min GPU / ~20 min CPU

In [ ]:
log.cell_start("Cell 5 — Training / Load Checkpoint")

import time
import numpy as np
from datetime import datetime

_PHASE6_FRESH_TRAIN = not checkpoint_exists(6)

if not _PHASE6_FRESH_TRAIN:
    print("⏩ Phase 6 checkpoint found — loading instead of training")
    ckpt6 = load_checkpoint(6)
    cfg = ckpt6.get('config', {})
    wave_dim = cfg.get('wave_dim', 432)
    feature_dim = cfg.get('feature_dim', 256)

    # Rebuild components from checkpoint
    working = WorkingMemory(
        window_size=cfg.get('window_size', 512),
        wave_dim=wave_dim,
        feature_dim=feature_dim,
    ).to(DEVICE)
    working.load_state_memory(ckpt6['working_memory_state'])
    working.eval()

    episodic = EpisodicMemory(feature_dim=feature_dim)
    episodic.load_state(ckpt6['episodic_memory_state'])

    semantic = SemanticMemory(field=field, protection_threshold=5.0).to(DEVICE)
    semantic.load_state(ckpt6['semantic_memory_state'])

    router = MemoryRouter(
        working=working, episodic=episodic, semantic=semantic,
        wave_dim=wave_dim, feature_dim=feature_dim,
    ).to(DEVICE)
    router.load_state(ckpt6['router_state'])

    consolidation = ConsolidationProcess(
        episodic=episodic, semantic=semantic, min_access=3,
    )

    metrics = ckpt6.get('metrics', {})
    print(f"  ✓ Phase 6 loaded from checkpoint")
    print(f"    Episodic accuracy: {metrics.get('episodic_accuracy', 'N/A')}")
    print(f"    Avg forgetting:    {metrics.get('avg_forgetting', 'N/A')}")
    print(f"    Field stability:   {metrics.get('field_stability', 'N/A')}")
    log.info("Phase 6 loaded from existing checkpoint")

else:
    print("── Starting Phase 6 Training ──\n")
    start_time = time.time()

    wave_dim = 432
    feature_dim = 256

    # ── Build components ──
    working = WorkingMemory(
        window_size=512, wave_dim=wave_dim, feature_dim=feature_dim,
    ).to(DEVICE)
    episodic = EpisodicMemory(feature_dim=feature_dim)
    semantic = SemanticMemory(field=field, protection_threshold=5.0).to(DEVICE)
    router = MemoryRouter(
        working=working, episodic=episodic, semantic=semantic,
        wave_dim=wave_dim, feature_dim=feature_dim,
    ).to(DEVICE)
    consolidation = ConsolidationProcess(
        episodic=episodic, semantic=semantic, min_access=3, temperature=0.05,
    )

    log.success(f"WorkingMemory built: window=512, {sum(p.numel() for p in working.parameters()):,} params")
    log.success(f"EpisodicMemory built: feature_dim={feature_dim}")
    log.success(f"SemanticMemory built: {semantic.num_protected} protected attractors")
    log.success(f"MemoryRouter built: {sum(p.numel() for p in router.parameters()):,} params")
    log.success("ConsolidationProcess built")

    # ════════════════════════════════════════════
    # Stage A: Working Memory Training
    # ════════════════════════════════════════════
    print("\n" + "=" * 65)
    print("  Stage A: Working Memory — Compression + Importance Scoring")
    print("=" * 65)

    optimizer_wm = torch.optim.Adam(working.parameters(), lr=1e-3)

    train_sentences = [
        "The capital of France is Paris",
        "Water boils at 100 degrees Celsius",
        "The earth orbits the sun",
        "Neural networks use backpropagation",
        "FLUX uses resonance fields instead of weights",
        "Semantic waves encode meaning continuously",
        "Gravitational relevance replaces attention",
        "Thermodynamic learning replaces backpropagation",
        "Causal geometry nodes store reasons for conclusions",
        "Memory consolidation happens during offline phases",
    ]

    waves = []
    for s in train_sentences:
        with torch.no_grad():
            w = cse.encode(s)
            waves.append(w.full.mean(dim=0).to(DEVICE))

    for epoch in range(50):
        total_loss = 0.0
        for wave in waves:
            compressed = working.compress(wave.unsqueeze(0))
            importance = working.importance_scorer(compressed)
            reconstructed = F.linear(compressed, working.compress.weight.T)
            loss = F.mse_loss(reconstructed, wave.unsqueeze(0))
            optimizer_wm.zero_grad()
            loss.backward()
            optimizer_wm.step()
            total_loss += loss.item()
        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1}/50  loss={total_loss / len(waves):.6f}")

    working.eval()
    for wave in waves:
        working.add_perturbation(wave)

    log.success(f"Working memory trained and populated: {working.size} entries")
    log.metric("wm_loss", f"{total_loss / len(waves):.6f}")

    # ════════════════════════════════════════════
    # Stage B: Episodic Memory — One-Shot Write/Read
    # ════════════════════════════════════════════
    print("\n" + "=" * 65)
    print("  Stage B: Episodic Memory — One-Shot Write/Read Accuracy")
    print("=" * 65)

    facts = [
        ("The capital of Mars colony Alpha is New Houston", "science_fiction"),
        ("FLUX processes raw bytes with no tokenization", "architecture"),
        ("Resonance fields replace weight matrices", "architecture"),
        ("Gravitational relevance costs O(log n)", "performance"),
        ("Thermodynamic learning has no epochs", "learning"),
        ("Causal nodes store WHY not just WHAT", "reasoning"),
        ("Working memory is session-scoped", "memory"),
        ("Episodic memory is permanent", "memory"),
        ("Semantic memory only updates offline", "memory"),
        ("Zero catastrophic forgetting by design", "key_feature"),
    ]

    for fact_text, source in facts:
        with torch.no_grad():
            wave = cse.encode(fact_text)
            vec = working.compress(wave.full.mean(dim=0).to(DEVICE).unsqueeze(0)).squeeze(0)
        episodic.write(vec, fact=fact_text, causal_source=source)

    correct = 0
    for fact_text, source in facts:
        with torch.no_grad():
            wave = cse.encode(fact_text)
            q_vec = working.compress(wave.full.mean(dim=0).to(DEVICE).unsqueeze(0)).squeeze(0)
        results = episodic.search(q_vec, k=1)
        if results and results[0][0].fact == fact_text:
            correct += 1

    episodic_accuracy = correct / len(facts)
    print(f"  Episodic retrieval accuracy: {correct}/{len(facts)} = {episodic_accuracy:.1%}")
    log.metric("episodic_accuracy", f"{episodic_accuracy:.4f}")

    # Simulate repeated access for consolidation candidates
    for _ in range(5):
        for fact_text, _ in facts[:5]:
            with torch.no_grad():
                wave = cse.encode(fact_text)
                q_vec = working.compress(wave.full.mean(dim=0).to(DEVICE).unsqueeze(0)).squeeze(0)
            episodic.search(q_vec, k=1)

    log.success(f"Episodic: {episodic.size} facts, {episodic_accuracy:.1%} accuracy")

    # ════════════════════════════════════════════
    # Stage C: Semantic Memory + Consolidation
    # ════════════════════════════════════════════
    print("\n" + "=" * 65)
    print("  Stage C: Semantic Memory — Protection + Consolidation")
    print("=" * 65)

    field_snapshot = semantic.get_field_snapshot()
    energy_before = semantic.get_field_energy()
    print(f"  Field energy before: {energy_before:.6f}")

    for i in range(5):
        semantic.protect_attractor(i)
    print(f"  Protected {semantic.num_protected} attractors")

    consolid_result = consolidation.run_consolidation(wave_dim=wave_dim)
    print(f"  Consolidated: {consolid_result['consolidated']} entries")
    print(f"  Stability: {consolid_result['stability']:.4f}")

    stability = semantic.compute_stability(field_snapshot)
    energy_after = semantic.get_field_energy()
    print(f"  Field stability: {stability:.4f}")
    print(f"  Field energy after: {energy_after:.6f}")

    log.metric("consolidation_entries", consolid_result['consolidated'])
    log.metric("field_stability", f"{stability:.4f}")
    log.success("Consolidation completed")

    # ════════════════════════════════════════════
    # Stage D: Memory Router
    # ════════════════════════════════════════════
    print("\n" + "=" * 65)
    print("  Stage D: Memory Router — Cross-Tier Integration")
    print("=" * 65)

    for qt in ["What is the capital of Mars?", "How does FLUX replace attention?", "Does FLUX forget?"]:
        with torch.no_grad():
            wave = cse.encode(qt)
            qw = wave.full.mean(dim=0).to(DEVICE)
        result = router.route_query(qw, episodic_k=3, working_k=5)
        n_ep = len(result['episodic_facts'])
        weights = result['tier_weights']
        print(f"  Query: '{qt}'")
        print(f"    Episodic: {n_ep} results | Weights: W={weights[0]:.3f} E={weights[1]:.3f} S={weights[2]:.3f}")

    log.success("Memory router integration verified")

    # ════════════════════════════════════════════
    # Stage E: Zero Forgetting
    # ════════════════════════════════════════════
    print("\n" + "=" * 65)
    print("  Stage E: Zero Forgetting Verification")
    print("=" * 65)

    task_pairs = [
        ("The sky is blue", "Water is wet"),
        ("Dogs are mammals", "Fish live in water"),
        ("Python is a language", "Java is also a language"),
        ("The sun is a star", "The moon reflects light"),
        ("FLUX uses fields", "Transformers use attention"),
        ("Paris is in France", "Tokyo is in Japan"),
        ("Iron is a metal", "Oxygen is a gas"),
        ("Math uses numbers", "Music uses notes"),
        ("Trees produce oxygen", "Cars produce CO2"),
        ("Books store knowledge", "Brains store memories"),
    ]

    forgetting_scores = []
    for task_a_text, task_b_text in task_pairs:
        with torch.no_grad():
            wave_a = cse.encode(task_a_text).full.mean(dim=0).to(DEVICE)
            vec_a = working.compress(wave_a.unsqueeze(0)).squeeze(0)
        episodic.write(vec_a, fact=task_a_text, causal_source="forgetting_test")
        results_before = episodic.search(vec_a, k=1)
        acc_before = 1.0 if (results_before and results_before[0][0].fact == task_a_text) else 0.0

        with torch.no_grad():
            wave_b = cse.encode(task_b_text).full.mean(dim=0).to(DEVICE)
            vec_b = working.compress(wave_b.unsqueeze(0)).squeeze(0)
        episodic.write(vec_b, fact=task_b_text, causal_source="forgetting_test")

        results_after = episodic.search(vec_a, k=1)
        acc_after = 1.0 if (results_after and results_after[0][0].fact == task_a_text) else 0.0
        forgetting = (acc_before - acc_after) / max(acc_before, 1e-8) if acc_before > 0 else 0.0
        forgetting_scores.append(forgetting)

    avg_forgetting = sum(forgetting_scores) / len(forgetting_scores)
    max_forgetting = max(forgetting_scores)
    print(f"  Average forgetting: {avg_forgetting:.6f}")
    print(f"  Max forgetting:     {max_forgetting:.6f}")
    print(f"  Target:             < 0.02 (2%)")
    print(f"  Result:             {'PASS ✓' if avg_forgetting < 0.02 else 'FAIL ✗'}")

    log.metric("avg_forgetting", f"{avg_forgetting:.6f}")
    if avg_forgetting < 0.02:
        log.success("Zero catastrophic forgetting verified!")
    else:
        log.error(f"Forgetting too high: {avg_forgetting:.4f}")

    elapsed = time.time() - start_time
    log.metric("training_time", f"{elapsed:.1f}s")
    log.success(f"Phase 6 training completed in {elapsed:.1f}s")

log.cell_end("Cell 5 — Training / Load Checkpoint", "PASS")

---
## Cell 6: Save Checkpoint & Upload to HuggingFace Hub

In [ ]:
log.cell_start("Cell 6 — Save & Upload Checkpoint")

from datetime import datetime

if not _PHASE6_FRESH_TRAIN:
    ckpt_path = Path('checkpoints/phase6.phase.pt')
    size_mb = ckpt_path.stat().st_size / 1e6 if ckpt_path.exists() else 0
    print(f"  ⏩ Checkpoint already saved — skipping save step")
    print(f"     Local:  {ckpt_path} ({size_mb:.1f} MB)")
    print(f"     Remote: https://huggingface.co/UnseenGAP/FLUX")
    log.info("Checkpoint already existed — save step skipped")
else:
    checkpoint_state = {
        'phase': 6,
        'timestamp': datetime.now().isoformat(),
        'config': {
            'wave_dim': wave_dim,
            'feature_dim': feature_dim,
            'window_size': 512,
            'cse': ckpt1.get('config', {}),
            'field': field_cfg,
        },
        'cse_state_dict': ckpt1['state_dict'],
        'field_state_dict': ckpt2['state_dict'],
        'phase5_state': {k: v for k, v in ckpt5.items() if k != 'state_dict'},
        'working_memory_state': working.state_dict_memory(),
        'episodic_memory_state': episodic.save_state(),
        'semantic_memory_state': semantic.save_state(),
        'router_state': router.save_state(),
        'metrics': {
            'training_time_seconds': elapsed,
            'episodic_accuracy': episodic_accuracy,
            'avg_forgetting': avg_forgetting,
            'max_forgetting': max_forgetting,
            'field_stability': stability,
            'consolidation_entries': consolid_result['consolidated'],
            'working_memory_size': working.size,
            'episodic_memory_size': episodic.size,
        },
    }
    save_checkpoint(6, checkpoint_state)

    ckpt_path = Path('checkpoints/phase6.phase.pt')
    if ckpt_path.exists():
        size_mb = ckpt_path.stat().st_size / 1e6
        log.success(f"Checkpoint saved: {ckpt_path} ({size_mb:.1f} MB)")
        print(f"  ✓ Checkpoint saved: {ckpt_path} ({size_mb:.1f} MB)")
    else:
        log.error("Checkpoint NOT found — save may have failed")

    uploaded = upload_checkpoint_to_hf(phase=6, hf_token=HF_TOKEN)
    if uploaded:
        log.success("Checkpoint uploaded to https://huggingface.co/UnseenGAP/FLUX")
        print("  ✓ Uploaded to HuggingFace Hub")
    else:
        log.warning("Checkpoint upload skipped — check HF_TOKEN")
        print("  ⚠ Upload skipped — no HF token")

    upload_logs_to_hf(phase=6, hf_token=HF_TOKEN)

log.cell_end("Cell 6 — Save & Upload Checkpoint", "PASS")

---
## Cells 7–9: Tests

| Test | Description | Pass Criteria |
|------|-------------|---------------|
| Test 1: Episodic Write/Read | One-shot write, retrieve after 100+ noise writes | Accuracy ≥ 90% |
| Test 2: Semantic Protection | Field unchanged after 1000 episodic writes | Stability ≥ 0.95 |
| Test 3: Forgetting Score | 10 sequential task pairs, measure degradation | Forgetting < 0.02 |

In [ ]:
log.cell_start("Cell 7 — Test 1: One-Shot Episodic Write/Read")

import os
_phase6_dir = str(Path.cwd() / 'phases' / 'phase6')
_orig_dir = os.getcwd()
os.chdir(_phase6_dir)

%run test_phase6_test1.py

os.chdir(_orig_dir)
log.cell_end("Cell 7 — Test 1: One-Shot Episodic Write/Read", "PASS")

In [ ]:
log.cell_start("Cell 8 — Test 2: Semantic Memory Protection")

os.chdir(_phase6_dir)

%run test_phase6_test2.py

os.chdir(_orig_dir)
log.cell_end("Cell 8 — Test 2: Semantic Memory Protection", "PASS")

In [ ]:
log.cell_start("Cell 9 — Test 3: Forgetting Score = 0.0")

os.chdir(_phase6_dir)

%run test_phase6_test3.py

os.chdir(_orig_dir)
log.cell_end("Cell 9 — Test 3: Forgetting Score = 0.0", "PASS")

---
## Cells 10–12: Demos

| Demo | Description |
|------|-------------|
| Demo 1: Cross-Session Memory | Facts survive session boundary without RAG or fine-tuning |
| Demo 2: Consolidation Live | Watch episodic → semantic distillation in action |
| Demo 3: Zero Forgetting × 1000 | 1000 sequential tasks, early task accuracy intact |

In [ ]:
log.cell_start("Cell 10 — Demo 1: Cross-Session Memory")

os.chdir(_phase6_dir)

%run demo_phase6_demo1.py

os.chdir(_orig_dir)
log.cell_end("Cell 10 — Demo 1: Cross-Session Memory", "PASS")

In [ ]:
log.cell_start("Cell 11 — Demo 2: Consolidation Process Live")

os.chdir(_phase6_dir)

%run demo_phase6_demo2.py

os.chdir(_orig_dir)
log.cell_end("Cell 11 — Demo 2: Consolidation Process Live", "PASS")

In [ ]:
log.cell_start("Cell 11b — Demo 3: Zero Forgetting over 1000 Tasks")

os.chdir(_phase6_dir)

%run demo_phase6_demo3.py

os.chdir(_orig_dir)
log.cell_end("Cell 11b — Demo 3: Zero Forgetting over 1000 Tasks", "PASS")

---
## Cell 12: Interactive Exploration

In [ ]:
log.cell_start("Cell 12 — Interactive Exploration")

# ── Interactive: Write custom facts and query them back ──
print("=" * 60)
print("  Interactive: Custom Memory Write/Read")
print("=" * 60)

# Load checkpoint if needed
if not checkpoint_exists(6):
    print("  ⚠ No checkpoint — run training first")
else:
    ckpt6 = load_checkpoint(6)
    cfg = ckpt6.get('config', {})

    # Rebuild fresh episodic for interactive demo
    em_interactive = EpisodicMemory(feature_dim=cfg.get('feature_dim', 256))

    # Custom facts
    custom_facts = [
        "My favorite color is deep ocean blue",
        "I built FLUX to prove new AI architectures are possible",
        "The best pizza topping is pineapple and jalapeno",
        "FLUX will eventually replace transformers",
        "Memory consolidation mimics human sleep",
    ]

    print("\n  Writing custom facts:")
    for fact in custom_facts:
        with torch.no_grad():
            wave = cse.encode(fact)
            vec = working.compress(wave.full.mean(dim=0).to(DEVICE).unsqueeze(0)).squeeze(0)
        em_interactive.write(vec, fact=fact, causal_source="interactive")
        print(f"    📝 {fact}")

    # Query
    custom_queries = [
        "What is my favorite color?",
        "What did I build FLUX for?",
        "Best pizza topping?",
    ]

    print("\n  Querying back:")
    for query in custom_queries:
        with torch.no_grad():
            q_wave = cse.encode(query)
            q_vec = working.compress(q_wave.full.mean(dim=0).to(DEVICE).unsqueeze(0)).squeeze(0)
        results = em_interactive.search(q_vec, k=2)
        print(f"\n    🔍 '{query}'")
        for entry, score in results:
            print(f"       → [{score:.3f}] {entry.fact}")

    # Print system stats
    print("\n" + "─" * 60)
    print("  Memory System Stats:")
    print(f"    Working memory entries: {working.size}")
    print(f"    Episodic entries (interactive): {em_interactive.size}")
    print(f"    Semantic protected attractors: {semantic.num_protected}")
    print(f"    Semantic field energy: {semantic.get_field_energy():.6f}")

    # Tier weight inspection
    weights = torch.softmax(router.tier_weights, dim=0)
    print(f"    Router tier weights: Working={weights[0]:.3f} Episodic={weights[1]:.3f} Semantic={weights[2]:.3f}")

torch.cuda.empty_cache() if torch.cuda.is_available() else None

log.cell_end("Cell 12 — Interactive Exploration", "PASS")

---
## Cell 13: View Results Summary & Full Log

In [ ]:
log.cell_start("Cell 13 — Results Summary & Log")

# ── Display RESULTS_PHASE_6.md ──
results_path = Path('phases/phase6/RESULTS_PHASE_6.md')
if results_path.exists():
    from IPython.display import Markdown, display
    display(Markdown(results_path.read_text()))
else:
    # Generate results if not yet created
    if checkpoint_exists(6):
        _ckpt = load_checkpoint(6)
        _m = _ckpt.get('metrics', {})
        results = PhaseResults(phase=6, component_name="Three-Tier Memory System")
        results.add_test("Episodic Write/Read",
                         _m.get('episodic_accuracy', 0) >= 0.9,
                         score=f"{_m.get('episodic_accuracy', 0):.4f}",
                         threshold="≥ 0.9")
        results.add_test("Semantic Protection",
                         _m.get('field_stability', 0) >= 0.95,
                         score=f"{_m.get('field_stability', 0):.4f}",
                         threshold="≥ 0.95")
        results.add_test("Zero Forgetting",
                         _m.get('avg_forgetting', 1) < 0.02,
                         score=f"{_m.get('avg_forgetting', 0):.6f}",
                         threshold="< 0.02")
        results.add_metric("training_time", f"{_m.get('training_time_seconds', 0):.1f}s")
        results.add_metric("episodic_entries", _m.get('episodic_memory_size', 0))
        results.add_metric("working_entries", _m.get('working_memory_size', 0))
        results.save()
        display(Markdown(results_path.read_text()))
    else:
        print("  ⚠ No results yet — run training first")

# ── Display full log ──
print("\n" + "=" * 60)
print("  FULL PHASE 6 LOG")
print("=" * 60)
print(log.get_log_contents())

log.cell_end("Cell 13 — Results Summary & Log", "PASS")

---
## Cell 14: Final Upload

In [ ]:
log.cell_start("Cell 14 — Final Upload")

upload_logs_to_hf(phase=6, hf_token=HF_TOKEN)
log.success("Logs uploaded to HuggingFace Hub")

git_commit_and_push(
    files=[
        'logs/phase6.log',
        'results/',
        'phases/phase6/RESULTS_PHASE_6.md',
    ],
    message='Phase 6: Three-Tier Memory System — training complete [auto-commit from Kaggle]',
    repo_dir=str(WORK_DIR),
)

log.cell_end("Cell 14 — Final Upload", "PASS")

print("\n" + "=" * 60)
print("✓ PHASE 6 COMPLETE")
print("=" * 60)
print(f"  Checkpoint: https://huggingface.co/UnseenGAP/FLUX")
print(f"  Logs:       https://huggingface.co/UnseenGAP/FLUX (logs/)")
print(f"  Code:       https://github.com/Unseengap/FLUX")
print(f"  Next:       Phase 7 — Full FLUX Integration")
print("=" * 60)

---
## Cell 15: Save Artifacts to Kaggle Output

In [ ]:
log.cell_start("Cell 15 — Save Kaggle Artifacts")

import shutil

output_dir = Path('/kaggle/working/output')
output_dir.mkdir(exist_ok=True)

# ── Checkpoints ──
for fname in ['phase6.phase.pt']:
    src = WORK_DIR / 'checkpoints' / fname
    if src.exists():
        shutil.copy2(str(src), str(output_dir / src.name))
        print(f"  ✓ Checkpoint: {src.name} ({src.stat().st_size/1e6:.1f} MB)")

# ── Results and logs ──
for fpath in [
    WORK_DIR / 'phases' / 'phase6' / 'RESULTS_PHASE_6.md',
    WORK_DIR / 'logs' / 'phase6.log',
]:
    if fpath.exists():
        shutil.copy2(str(fpath), str(output_dir / fpath.name))
        print(f"  ✓ {fpath.name}")

# ── List all saved artifacts ──
print("\n  Saved artifacts:")
for f in sorted(output_dir.iterdir()):
    size = f.stat().st_size
    unit = 'MB' if size > 1e6 else 'KB'
    val = size / 1e6 if size > 1e6 else size / 1e3
    print(f"    {f.name:40s} {val:8.1f} {unit}")

log.cell_end("Cell 15 — Save Kaggle Artifacts", "PASS")

print("\n" + "=" * 60)
print("ALL DONE — Phase 6: Three-Tier Memory System")
print("=" * 60)

---
## Acceptance Criteria

| Criterion | Target | Method |
|-----------|--------|--------|
| Episodic write → retrieve after 100+ noise writes | Accuracy ≥ 90% | Test 1 |
| Semantic field unchanged after 1000 episodic writes | Stability ≥ 0.95 | Test 2 |
| Forgetting score across 10 task pairs | < 0.02 (2%) | Test 3 |
| Consolidation promotes frequent episodic → semantic | Verified | Stage C |
| Memory persists across save/load cycle | Verified | Demo 1 |
| All tests pass | 3/3 | Cells 7-9 |

### Key Design Decisions

- **Working Memory** uses importance-weighted eviction, not FIFO — more relevant entries persist longer
- **Episodic Memory** uses normalized inner-product FAISS index for O(1) write + O(log n) search
- **Semantic Memory** wraps the ResonanceField with energy barriers — the field IS semantic memory
- **Consolidation** uses low-temperature thermodynamic settling — gentle integration without disruption
- **Zero forgetting** is architectural, not algorithmic — episodic store is additive, never overwrites

### Memory Tier Summary

| Tier | Capacity | Persistence | Speed | Update Method |
|------|----------|-------------|-------|---------------|
| Working | 512 entries | Session only | Immediate | Rolling window |
| Episodic | Unlimited | Permanent | FAISS search | One-shot write |
| Semantic | Field state | Permanent | Always on | Consolidation only |

# Phase 6: Three-Tier Memory System
## Working, Episodic, and Semantic Memory Integration